In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# Modelo que quieres usar
MODEL = "gemini-2.5-flash-lite"

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

In [ ]:
loader = PyPDFDirectoryLoader("../data/contratos_laborales")
documentos = loader.load()

print(f"Se han cargado {len(documentos)} páginas")

Se han cargado 6 páginas


In [ ]:
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/embedding-001",
#     google_api_key=API_KEY
# )

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=1000
)

docs_split = text_splitter.split_documents(documentos)

print(f"Se crearon {len(docs_split)} chunks de texto.")

Se crearon 11 chunks de texto.


In [ ]:
vectorstore = Chroma.from_documents(
    docs_split,
    embedding=embeddings,
    persist_directory="../data/chroma_db"
)

In [ ]:
consulta = "Cuanto pagará el empleador al trabajador JULIÁN MIGUEL TORRES CALVO segun se indica en QUINTA. Retribución"

resultados = vectorstore.similarity_search(consulta, k=4)

print("Top 1 documentos mas similares a la consulta:\n")
for i, doc in enumerate(resultados, start=4):
    print(f"Contenido: {doc.page_content}")
    print(f"Metadatos: {doc.metadata}")

Top 1 documentos mas similares a la consulta:

Contenido: DNI: 62491837-T.
Nacionalidad: Española.
Fecha de nacimiento: 11 de junio de 1993.
Estado civil: Soltero.
Domicilio: Calle Jardines del Turia nº 18, Piso 4, Puerta A, 46010, Valencia, España.
Correo electrónico: julian.torres@atlantis-ejemplo.es
Teléfono: +34 655 772 309.
II. DECLARACIONES
A. Declara el Empleador:
Que está legalmente constituido y facultado para contratar personal.
Que necesita reforzar su equipo técnico especializado.
B. Declara el Trabajador:
Que posee la cualificación técnica adecuada para el puesto.
Que acepta voluntariamente la relación laboral en régimen de subordinación.
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
1. 
2. 
1. 
2. 
1
Metadatos: {'producer': 'WeasyPrint 65.1', 'page': 0, 'total_pages': 3, 'author': 'ChatGPT Canvas', 'title': 'Contrato Laboral Individual Iv', 'page_label': '1', 'creationdate': '', 'creator': 'ChatGPT', 'source': '..\\data\\contratos_laborales\\Contrato Laboral Individual II.